# Increase Resolution — capabilities walkthrough

This notebook runs **Increase Resolution** end to end: install the **`increase-resolution`** package from **AWS CodeArtifact**, download the Bria super-resolution **TensorRT engine(s)**, then upscale an image **2×** and **4×** with the in-process tiling pipeline.

**Required environment (not flexible):** NVIDIA **A10** (Ampere), **CUDA 11.7.1**, **TensorRT 8.4.1.5** (exact patch), Python **3.10**. The exact CUDA 11.7.1 libraries are not pip-installable, so use the NVIDIA base image `nvcr.io/nvidia/pytorch:22.07-py3` (CUDA 11.7.1 + TensorRT 8.4.1.5) with Python 3.10. **Before running this notebook**, prepare the runtime as described in the README's *Runtime setup* section (base image + `unset PYTHONPATH`/`LD_LIBRARY_PATH`, `torch==2.0.1+cu117`, `nvidia-tensorrt==8.4.1.5`, `numpy<2`, and the cuBLAS `LD_PRELOAD`). A TensorRT engine only loads on the GPU/runtime it was built for.

**Prerequisites:** `BRIA_API_TOKEN` (for the CodeArtifact credential).

## 1. CodeArtifact token (Bria Engine)

Request a short-lived credential for the **`bria-increase-resolution`** repository.

In [ ]:
import os
import requests

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN")
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN before running this notebook.")

resp = requests.get(
    "https://engine.prod.bria-api.com/v2/auth/access/code_artifact",
    params={"repository": "bria-increase-resolution"},
    headers={"api_token": BRIA_API_TOKEN},
    timeout=60,
)
resp.raise_for_status()
result = resp.json()["result"]
CODE_ARTIFACT_PASSWORD = result["authorization_token"].strip()
print("CodeArtifact credential acquired. Expires:", result.get("expiration"))

## 2. Install `increase-resolution`

Uses the CodeArtifact token as the password for the `bria-increase-resolution` PyPI index (username `aws`). `torch` and `tensorrt==8.4.*` must already match your CUDA 11.7 environment.

In [ ]:
import subprocess, sys
from urllib.parse import quote

index = (
    "https://aws:" + quote(CODE_ARTIFACT_PASSWORD, safe="")
    + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-increase-resolution/simple/"
)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade",
                       "increase-resolution", "--extra-index-url", index])

## 3. Download the super-resolution engine(s)

Bria provides the engines as `.engine` files. Replace the URLs below with the download links Bria gave you. They are fetched over plain https into a local cache folder.

In [ ]:
from pathlib import Path

cache = Path(os.path.expanduser("~/.cache/bria/increase-resolution/"))
cache.mkdir(parents=True, exist_ok=True)

ENGINE_URLS = {
    2: "<BRIA_PROVIDED_URL_FOR_increase_resolution2.engine>",
    4: "<BRIA_PROVIDED_URL_FOR_increase_resolution4.engine>",
}
engine_paths = {}
for scale, url in ENGINE_URLS.items():
    dest = cache / f"increase_resolution{scale}.engine"
    if not dest.exists():
        # download to a temp file and rename on success, so an interrupted
        # download never leaves a truncated .engine that fails at setup()
        tmp = dest.with_suffix(".engine.part")
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
        tmp.rename(dest)
    engine_paths[scale] = str(dest)
    print(f"x{scale} engine:", dest)

## 4. Build configuration and start the pipeline

In [ ]:
import torch
from IPython.display import display

from increase_resolution import IncreaseResolution, IncreaseResolutionConfig, IncreaseResolutionInput

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Increase Resolution.")

config = IncreaseResolutionConfig(engine_paths={int(k): v for k, v in engine_paths.items()})
pipeline = IncreaseResolution(config=config)
pipeline.setup()
print("Pipeline setup complete.")

## 5. Upscale a sample image (2× and 4×)

In [ ]:
import time
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)

SAMPLE = "https://bria-test-images.s3.us-east-1.amazonaws.com/highway.jpg"

t0 = time.time()
out2 = pipeline.execute(IncreaseResolutionInput(image=SAMPLE, desired_increase=2))
print(f"2x: {out2.image.size} in {time.time()-t0:.1f}s")
out2.image.save("outputs/upscaled_x2.png")
display(out2.image)

In [ ]:
t0 = time.time()
out4 = pipeline.execute(IncreaseResolutionInput(image=SAMPLE, desired_increase=4))
print(f"4x: {out4.image.size} in {time.time()-t0:.1f}s")
out4.image.save("outputs/upscaled_x4.png")
display(out4.image)

## 6. Cleanup

In [ ]:
pipeline.cleanup()
print("Cleanup done.")